# Run the simulation

In [ ]:
# Jupyter-compatible version of run.py

from model import OpenCoverModel
from mesa.batchrunner import batch_run
import pickle
import pandas as pd
from tqdm import tqdm

# === Simulation Parameters (replace argparse with manual assignment) ===
behavior = "my_behavior"  # options: "pure_levy", "truncated_levy", "composite_levy", "biased_composite", "my_behavior", "all"
landing_prob = 1.0
n_agents = 60
iterations = 5           # Number of repetitions per setting (each with a different seed)
steps = 2 * 60 * 60       # Total number of simulation steps
num_processes = 10        # Use None for all CPUs

# === Determine which behaviors to run ===
if behavior == "all":
    behaviors_to_run = ["pure_levy", "truncated_levy", "composite_levy", "biased_composite", "my_behavior"]
else:
    behaviors_to_run = [behavior]

# === Run simulations ===
for behavior in behaviors_to_run:
    print(f"\n=== Running behavior: {behavior} ===\n")

    # Batch parameters
    param_dict = {
        "n_agents": n_agents,
        "bee_behavior": behavior,
        "landing_prob": landing_prob,
        "seed": range(iterations)
    }

    results = batch_run(
        model_cls=OpenCoverModel,
        parameters=param_dict,
        iterations=1,
        max_steps=steps,
        number_processes=num_processes,
        data_collection_period=10,
        display_progress=True
    )

    # Save output
    df = pd.DataFrame(results)
    output_filename = f"batch_results_{behavior}.pkl"
    with open(output_filename, "wb") as f:
        pickle.dump(results, f)

    print(f"Simulation complete. Results saved to {output_filename}")


# Animating average number of bee visits per column 

In [ ]:
%matplotlib notebook
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from IPython.display import HTML
from datetime import datetime, timedelta

# Step 1: Load the data
with open("batch_results_my_behavior.pkl", "rb") as f:
    raw_data = pickle.load(f)

df = pd.DataFrame(raw_data)

# Step 2: Compute average bee visits from timestamp queues
def compute_avg_visits_per_column_per_10min(df, visit_window_minutes=5, interval=600):
    """
    Every `interval` steps (e.g., 600s = 10 min), compute:
    - Total visits in past 5 minutes (from already-trimmed visit timestamp)
    - Average per column (across 5 simulations)
    - Divided by 5 to get visits per minute
    
    Returns:
        avg_visits_df: DataFrame with ['Step', 'x', 'avg_visits']
    """
    visit_window = 60 * visit_window_minutes
    plant_df = df[df["Agent Type"] == "plant"].copy()
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])

    all_steps = sorted(plant_df["Step"].unique())
    all_runs = sorted(plant_df["RunId"].unique())

    steps_to_plot = [s for s in all_steps if s % interval == 0]
    results = []

    print("Computing 5-min average visits per column every 10 min...")
    for step in tqdm(steps_to_plot):
        for run in all_runs:
            plants_this_step = plant_df[(plant_df["Step"] == step) & (plant_df["RunId"] == run)].copy()

            # Simply count how many timestamps are retained (i.e., visited in past 5 min)
            plants_this_step["recent_visits"] = plants_this_step["Plant visit timestamp"].apply(
                lambda ts: len(ts) if isinstance(ts, list) else 0
            )

            visits_per_x = (
                plants_this_step.groupby("x")["recent_visits"]
                .sum()
                .reset_index()
            )
            visits_per_x["RunId"] = run
            visits_per_x["Step"] = step
            results.append(visits_per_x)

    # Combine and average across simulations
    full_df = pd.concat(results, ignore_index=True)

    # Compute average visits per column and divide by 5 (to get per-minute average over 5-min window)
    avg_visits_df = (
        full_df.groupby(["Step", "x"])["recent_visits"]
        .mean()
        .reset_index()
        .rename(columns={"recent_visits": "avg_visits"})
    )
    avg_visits_df["avg_visits"] = avg_visits_df["avg_visits"] / visit_window_minutes

    return avg_visits_df

# Compute it
avg_visits_df = compute_avg_visits_per_column_per_10min(df)

In [ ]:
# Step 3: Animate
def animate_avg_visits(avg_visits_df, interval=600, start_time_seconds=0):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")

    end_time_seconds = avg_visits_df["Step"].max()
    total_frames = (end_time_seconds - start_time_seconds) // interval
    x_vals = sorted(avg_visits_df["x"].unique())
    global_ymax = max(avg_visits_df["avg_visits"].max() * 1.1, 1e-3)

    def update(frame_idx):
        ax.clear()
        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        df_frame = avg_visits_df[avg_visits_df["Step"] == current_time_seconds]

        sns.lineplot(data=df_frame, x="x", y="avg_visits", ax=ax, marker="o")

        ax.set_ylim(0, global_ymax)
        ax.set_title(f"Avg Bee Visits per Column at {time_str_for_title}")
        ax.set_xlabel("Polytunnel column x-position")
        ax.set_ylabel("Avg Bee Visits per Minute (past 5 min avg, across 5 runs)")
        ax.grid(True)

    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())



In [ ]:
%matplotlib notebook
# Step 4: Run animation
anim, html = animate_avg_visits(avg_visits_df, interval=600)
html

In [ ]:
df[df["Agent Type"] == "plant"]["Plant visit timestamp"].apply(
    lambda ts: len(ts) if isinstance(ts, list) else 0
).describe()


In [ ]:
step_to_check = df["Step"].max()
df_check = df[(df["Agent Type"] == "plant") & (df["Step"] == step_to_check)].copy()

df_check["x"] = df_check["Bee Positions"].apply(lambda pos: pos[0])
df_check["visit_count"] = df_check["Plant visit timestamp"].apply(lambda ts: len(ts) if isinstance(ts, list) else 0)

df_check[["x", "visit_count"]].groupby("x").sum().plot(
    kind="bar", figsize=(10,4), title=f"Total visits per column at step {step_to_check}"
)


# ---------------------------------------------------

# Animating Bee Visitation Rate in a minute at different time intervals

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
from IPython.display import HTML

In [ ]:
%matplotlib notebook
def load_dataset(filepath):
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60
    window_minutes = 1.0

    plant_df = df[
        (df["Agent Type"] == "plant") & (df["Step"] >= t_start) & (df["Step"] < t_end)
    ].copy()

    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(eval)
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )

    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )

    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes
    return visits_per_x_run

def precompute_visit_counts(interval_data_dict, df_sample, interval=600):
    precomputed_visit_counts = {}
    all_visits = []

    start_time_seconds = 0
    end_time_seconds = df_sample["Step"].max() + 1

    for label, df in interval_data_dict.items():
        precomputed_visit_counts[label] = {}
        for current_time_seconds in range(start_time_seconds, end_time_seconds, interval):
            current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
            current_time_str = current_time.strftime("%H:%M:%S")
            visit_counts = extract_visit_data(df, current_time_str)
            precomputed_visit_counts[label][current_time_seconds] = visit_counts
            all_visits.extend(visit_counts["visits_per_minute"].values)

    global_ymin = 0
    global_ymax = max(all_visits) * 1.1 if all_visits else 1

    return precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax

def animate_visits_precomputed(precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")
    keys_list = list(precomputed_visit_counts.keys())
    num_simulations = len(next(iter(precomputed_visit_counts[keys_list[0]].values()))["RunId"].unique())

    def update(frame_idx):
        ax.clear()
        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for label in precomputed_visit_counts.keys():
            visit_counts = precomputed_visit_counts[label][current_time_seconds]
            sns.lineplot(data=visit_counts, x="x", y="visits_per_minute", errorbar=("ci", 95), label=label, ax=ax)

        ax.set_ylim(global_ymin, global_ymax)
        ax.set_title(
            f"Mean number of bee visits in one-minute window starting at {time_str_for_title} across {num_simulations} simulations (95% CI)"
        )
        ax.set_xlabel("Polytunnel length (m)")
        ax.set_ylabel("Number of Bee Visits in One Minute Window")
        ax.grid(True)
        ax.legend()

    total_frames = (end_time_seconds - start_time_seconds) // interval
    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())


In [ ]:
# Load datasets
# df_pure = load_dataset("batch_results_pure_levy.pkl")
# df_truncated = load_dataset("batch_results_truncated_levy.pkl")
# df_composite = load_dataset("batch_results_composite_levy.pkl")
# df_follow = load_dataset("batch_results_biased_composite.pkl")
df_mybehavior = load_dataset("batch_results_my_behavior.pkl")
interval_data_dict = {
    # "pure_levy": df_pure,
    # "truncated_levy": df_truncated,
    # "composite_levy": df_composite,
    # "biased_composite_levy": df_follow,
    "my_bee_behavior": df_mybehavior 
}

# Precompute 10-minute averages
precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(interval_data_dict, df_mybehavior, interval=600)

# del df_pure, df_truncated, df_composite  # Free memory
del df_mybehavior  # Free memory


# produce the animation
anim, html_obj = animate_visits_precomputed(
    precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax, interval=600
)
html_obj

In [ ]:
# Saving the animation to HTML file
with open("bee_visitation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")

# Animate Nectar amount and bee visitation rate at different intervals of the simulation

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
from IPython.display import HTML

In [ ]:
def load_dataset(filepath):
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60  # 60-second window
    window_minutes = 1.0

    plant_df = df[
        (df["Agent Type"] == "plant") & (df["Step"] >= t_start) & (df["Step"] < t_end)
    ].copy()

    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        import ast
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(ast.literal_eval)

    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")
    plant_df["Plant nectar amount"] = pd.to_numeric(plant_df["Plant nectar amount"], errors="coerce")
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    # Compute bee visit counts
    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )
    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )
    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes

    # Total nectar per column (not per plant), per simulation
    nectar_per_column = (
        plant_df.groupby(["RunId", "x"])["Plant nectar amount"]
        .sum()
        .reset_index()
        .rename(columns={"Plant nectar amount": "total_nectar_per_column"})
    )

    return visits_per_x_run, nectar_per_column

def precompute_visit_counts(interval_data_dict, df_sample, interval=600):
    precomputed_data = {}
    all_visits = []

    start_time_seconds = 0
    end_time_seconds = df_sample["Step"].max() + 1

    for label, df in interval_data_dict.items():
        precomputed_data[label] = {}
        for current_time_seconds in range(start_time_seconds, end_time_seconds, interval):
            current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
            current_time_str = current_time.strftime("%H:%M:%S")
            visit_df, nectar_df = extract_visit_data(df, current_time_str)
            # Normalize nectar by number of 10s intervals in 10 minutes (i.e., 600s / 10s = 60)
            nectar_df["total_nectar_per_column"] /= 6
            precomputed_data[label][current_time_seconds] = (visit_df, nectar_df)
            all_visits.extend(visit_df["visits_per_minute"].values)

    global_ymin = 0
    global_ymax = max(all_visits) * 1.1 if all_visits else 1

    return precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax

def animate_visits_with_nectar(precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")

    ax2 = ax1.twinx()
    keys_list = list(precomputed_data.keys())
    num_simulations = len(next(iter(precomputed_data[keys_list[0]].values()))[0]["RunId"].unique())

    def update(frame_idx):
        ax1.clear()
        ax2.clear()

        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for i, label in enumerate(precomputed_data.keys()):
            visit_df, nectar_df = precomputed_data[label][current_time_seconds]

            # Plot visitation rate (left axis)
            sns.lineplot(data=visit_df, x="x", y="visits_per_minute", ax=ax1, label=f"{label} visits")

            # Plot average nectar (right axis, normalized by 6)
            nectar_avg = (
                nectar_df.groupby("x")["total_nectar_per_column"]
                .mean()
                .reset_index()
            )

            offset = (i - 0.5) * 0.4
            ax2.bar(nectar_avg["x"] + offset,
                    nectar_avg["total_nectar_per_column"],
                    width=0.6,
                    alpha=0.3,
                    label=f"{label} nectar")

        ax1.set_ylim(global_ymin, global_ymax)
        ax1.set_title(f"Bee Visits and Nectar Levels at {time_str_for_title} across {num_simulations} simulations")
        ax1.set_xlabel("Polytunnel length (m)")
        ax1.set_ylabel("Bee Visits / min")
        ax2.set_ylabel("Avg Nectar per Column (normalized)")

        ax2.yaxis.set_label_position("right")
        ax2.yaxis.tick_right()
        ax1.tick_params(axis='y', labelcolor='black')
        ax2.tick_params(axis='y', labelcolor='black')
        ax1.grid(True)
        ax1.legend(loc="upper left")
        ax2.legend(loc="upper right")

    total_frames = (end_time_seconds - start_time_seconds) // interval
    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())


In [ ]:
# Load datasets
# df_follow = load_dataset("batch_results_biased_composite.pkl")
df_mybehavior = load_dataset("batch_results_my_behavior.pkl")

interval_data_dict = {
    # "biased_composite_levy": df_follow,
    "my_bee_behavior": df_mybehavior
}

# Precompute at 10-minute intervals (600 seconds)
precomputed_data, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(
    interval_data_dict, df_mybehavior, interval=600
)

# Free memory
# del df_follow, df_mybehavior

# Display animation
anim, html_obj = animate_visits_with_nectar(
    precomputed_data, start_sec, end_sec, global_ymin, global_ymax, interval=600
)
html_obj


In [ ]:
# Load datasets
rep_rate = "100"
df_mybehavior = load_dataset(f"batch_results_my_behavior_rep_{rep_rate}.pkl")

interval_data_dict = {
    f"Replenish Rate {rep_rate}": df_mybehavior
}

# Precompute at 10-minute intervals (600 seconds)
precomputed_data, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(
    interval_data_dict, df_mybehavior, interval=600
)

# Free memory
# del df_follow, df_mybehavior

# Display animation
anim, html_obj = animate_visits_with_nectar(
    precomputed_data, start_sec, end_sec, global_ymin, global_ymax, interval=600
)
html_obj


In [ ]:
# Saving the animation to HTML file
with open(f"bee_visitation_animation_{rep_rate}.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")

# Plot all in one (Not working yet)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
import pandas as pd

def animate_visits_shared_slider(precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    """
    Create a single animation with subplots (one per dataset) that share the same slider.
    """
    sns.set_theme(style="whitegrid")
    labels = list(precomputed_data.keys())
    num_datasets = len(labels)
    total_frames = (end_time_seconds - start_time_seconds) // interval

    # Create subplots: one row per dataset
    fig, axes = plt.subplots(nrows=num_datasets, ncols=1, figsize=(12, 5 * num_datasets), sharex=True)
    if num_datasets == 1:
        axes = [axes]

    def update(frame_idx):
        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for ax, label in zip(axes, labels):
            ax.clear()
            ax2 = ax.twinx()

            visit_df, nectar_df = precomputed_data[label][current_time_seconds]

            sns.lineplot(data=visit_df, x="x", y="visits_per_minute", ax=ax, color="tab:blue", label="Visits")
            nectar_avg = (
                nectar_df.groupby("x")["total_nectar_per_column"]
                .mean()
                .reset_index()
            )
            ax2.bar(nectar_avg["x"], nectar_avg["total_nectar_per_column"],
                    width=0.6, alpha=0.3, color="tab:orange", label="Nectar")

            ax.set_ylim(global_ymin, global_ymax)
            ax.set_ylabel("Visits/min", color="tab:blue")
            ax2.set_ylabel("Avg Nectar", color="tab:orange")
            ax.set_title(f"{label} — {time_str_for_title}")

            ax2.yaxis.set_label_position("right")
            ax2.yaxis.tick_right()
            ax.tick_params(axis='y', labelcolor='tab:blue')
            ax2.tick_params(axis='y', labelcolor='tab:orange')
            ax.grid(True)

            h1, l1 = ax.get_legend_handles_labels()
            h2, l2 = ax2.get_legend_handles_labels()
            ax2.legend(h1 + h2, l1 + l2, loc="upper right")

    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    from IPython.display import HTML
    return anim, HTML(anim.to_jshtml())



In [ ]:
# Load datasets
rep000 = load_dataset("batch_results_my_behavior_rep_000.pkl")
rep005 = load_dataset("batch_results_my_behavior_rep_005.pkl")
rep100 = load_dataset("batch_results_my_behavior_rep_100.pkl")
interval_data_dict = {
    "Replenish rate 0": rep000,
    "Replenish rate 0.05": rep005,
    "Replenish rate 1.00": rep100,
}

# Precompute at 10-minute intervals (600 seconds)
precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax = precompute_visit_counts(
    interval_data_dict, df_sample=rep000, interval=600
)

# Free memory
del rep000, rep005, rep100


In [ ]:
anim, html = animate_visits_shared_slider(
    precomputed_data,
    start_time_seconds,
    end_time_seconds,
    global_ymin,
    global_ymax,
    interval=600
)
html

In [ ]:
# Saving the animation to HTML file
with open("bee_visitation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")